# Atribución Multicanal — Olist Marketing Funnel
## CRISP-DM: Fases 1, 2 y 3

| Parámetro | Valor |
|---|---|
| **Dataset** | Olist Marketing Funnel (Kaggle) |
| **Periodo MQLs** | 14 Jun 2017 – 31 May 2018 |
| **Periodo Deals** | 05 Dic 2017 – 14 Nov 2018 |
| **Objetivo** | Atribución multicanal con Cadenas de Markov para el funnel B2B de Olist |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})
COLORS = sns.color_palette('Set2')
sns.set_palette('Set2')

---
# Fase 1: Business Understanding

> **Objetivo:** Definir qué quiere responder el negocio antes de tocar datos.

Antes de cualquier análisis documentamos las decisiones de negocio que enmarcan el modelo.

## 1.1 Definición de Conversión

En el contexto del dataset de Olist Marketing Funnel, una **conversión** se define como:

> Un MQL (`mql_id`) que aparece en `olist_closed_deals_dataset.csv`, indicando que el prospecto fue captado como seller activo en la plataforma Olist.

**Criterio técnico:** `mql_id` presente en `closed_deals` con `won_date` registrada.

## 1.2 Canales de Marketing Disponibles

El dataset registra el canal de primer (y único) contacto en la columna `origin`. Los canales se identificarán en la celda siguiente.

In [ ]:
# Cargar datos para identificar canales
mql   = pd.read_csv('olist_marketing_qualified_leads_dataset.csv', parse_dates=['first_contact_date'])
deals = pd.read_csv('olist_closed_deals_dataset.csv', parse_dates=['won_date'])

# Limpiar origin vacío
mql['origin'] = mql['origin'].fillna('(not set)').replace('', '(not set)')

channels_info = (
    mql.groupby('origin').size()
    .reset_index(name='MQLs')
    .sort_values('MQLs', ascending=False)
    .assign(pct_total=lambda x: (x['MQLs'] / len(mql) * 100).round(2))
)
channels_info.columns = ['Canal', 'MQLs', '% del Total']
display(channels_info.to_string(index=False))

## 1.3 Horizonte Temporal del Análisis

| Concepto | Rango |
|---|---|
| Generación de MQLs | 14 Jun 2017 – 31 May 2018 (≈ 11.5 meses) |
| Cierre de deals | 05 Dic 2017 – 14 Nov 2018 |
| Ventana de conversión | Sin límite superior fijo en el dataset |

## 1.4 Reglas de Negocio

| Regla | Decisión | Justificación |
|---|---|---|
| Leads con tiempo negativo (`won_date` < `first_contact`) | **Mantener con flag** | Posible error de captura; se revisará en Fase 5 |
| Leads que tardan > 90 días en convertir | **Incluir con flag `long_journey`** | Representan ~17.8 % de conversiones; excluirlos sesgaría el modelo |
| `origin` vacío (60 MQLs) | **Etiquetar como `(not set)`** | Canal real desconocido, diferente del canal `unknown` |
| Canal `unknown` (1 099 MQLs) | **Mantener como categoría** | Tráfico trazado con fuente no identificada |

## 1.5 Métricas de Éxito

1. **Removal Effect por canal** (Markov): % de conversiones que se pierden al eliminar cada canal
2. **Ranking relativo de canales**: ¿cambia el orden respecto a atribución simple por tasa de conversión?
3. **Coherencia:** suma de conversiones atribuidas == 842 (conversiones reales)
4. **Estabilidad:** resultados estables al cambiar el periodo de análisis (Fase 5)

---

## ✅ Entregable Fase 1

- **Conversión:** MQL con `won_date` registrada en `closed_deals`
- **Canales:** 10 categorías + `(not set)`
- **Horizonte:** Jun 2017 – Nov 2018
- **Regla de 90 días:** incluir con flag
- **Métrica principal:** Removal Effect por canal

---
# Fase 2: Data Understanding

> **Objetivo:** Explorar el dataset y entender su estructura real antes de modelar.

## 2.1 Estructura de los Datasets

In [ ]:
print('=' * 55)
print('MQL DATASET')
print('=' * 55)
print(f'Shape: {mql.shape}')
print(f'\nDtypes:\n{mql.dtypes}')
print('\nPrimeras filas:')
display(mql.head())

print('\n' + '=' * 55)
print('CLOSED DEALS DATASET')
print('=' * 55)
print(f'Shape: {deals.shape}')
print(f'\nDtypes:\n{deals.dtypes}')
print('\nPrimeras filas:')
display(deals.head())

## 2.2 Análisis de Valores Nulos y Vacíos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

def plot_nulls(df, ax, title):
    # Tratar strings vacíos también como nulos
    null_pct = df.apply(lambda col: (col.isnull() | col.astype(str).eq('')).mean() * 100)
    null_pct = null_pct[null_pct > 0].sort_values()
    if null_pct.empty:
        ax.text(0.5, 0.5, 'Sin valores nulos', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
        ax.set_title(title)
        return
    bars = ax.barh(null_pct.index, null_pct.values, color=COLORS[2])
    for bar, val in zip(bars, null_pct.values):
        ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=9)
    ax.set_xlabel('% nulos o vacíos')
    ax.set_title(title)
    ax.set_xlim(0, 115)

plot_nulls(mql, axes[0], 'Nulos — MQL Dataset')
plot_nulls(deals, axes[1], 'Nulos — Closed Deals Dataset')
plt.tight_layout()
plt.savefig('fig_nulls.png', dpi=120, bbox_inches='tight')
plt.show()

print('Nota: has_company, has_gtin, average_stock y declared_product_catalog_size'
      ' tienen >90 % vacíos — no se usarán para el modelo de atribución.')

## 2.3 Distribución de MQLs por Canal

In [ ]:
channel_counts = mql['origin'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(channel_counts.index, channel_counts.values, color=COLORS)
ax.set_title('Distribución de MQLs por Canal de Origen')
ax.set_xlabel('Canal')
ax.set_ylabel('Número de MQLs')
for bar, val in zip(bars, channel_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 25,
        f'{val:,}\n({val/len(mql)*100:.1f}%)',
        ha='center', va='bottom', fontsize=8.5
    )
plt.tight_layout()
plt.savefig('fig_channel_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.4 Tasa de Conversión por Canal

In [ ]:
deal_ids = set(deals['mql_id'])
mql['converted'] = mql['mql_id'].isin(deal_ids)

conv_by_ch = (
    mql.groupby('origin')
    .agg(MQLs=('mql_id', 'count'), Conversiones=('converted', 'sum'))
    .assign(Tasa_Conv_pct=lambda x: x['Conversiones'] / x['MQLs'] * 100)
    .sort_values('Tasa_Conv_pct', ascending=False)
)

display(
    conv_by_ch.style.format({'MQLs': '{:,}', 'Conversiones': '{:,}', 'Tasa_Conv_pct': '{:.2f}%'})
)

global_rate = mql['converted'].mean() * 100
colors_bar = [COLORS[0] if v > global_rate else COLORS[2] for v in conv_by_ch['Tasa_Conv_pct']]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(conv_by_ch.index, conv_by_ch['Tasa_Conv_pct'], color=colors_bar)
ax.axhline(y=global_rate, color='red', linestyle='--', linewidth=1.5, alpha=0.8,
           label=f'Media global: {global_rate:.2f}%')
ax.set_title('Tasa de Conversión por Canal (verde = sobre la media)')
ax.set_xlabel('Canal')
ax.set_ylabel('Tasa de Conversión (%)')
ax.legend()
for bar, val in zip(bars, conv_by_ch['Tasa_Conv_pct']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.15,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('fig_conversion_rates.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.5 Volumen vs. Tasa de Conversión

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for i, (channel, row) in enumerate(conv_by_ch.iterrows()):
    size = max(row['Conversiones'] * 8, 50)
    ax.scatter(row['MQLs'], row['Tasa_Conv_pct'], s=size, alpha=0.75,
               color=COLORS[i % len(COLORS)], edgecolors='white', linewidth=2, zorder=3)
    ax.annotate(channel, (row['MQLs'], row['Tasa_Conv_pct']),
                textcoords='offset points', xytext=(8, 2), fontsize=9)

ax.axhline(y=global_rate, color='red', linestyle='--', alpha=0.6,
           label=f'Media: {global_rate:.2f}%')
ax.set_xlabel('Volumen de MQLs generados')
ax.set_ylabel('Tasa de Conversión (%)')
ax.set_title('Volumen vs. Tasa de Conversión por Canal\n(tamaño = conversiones absolutas)')
ax.legend()
plt.tight_layout()
plt.savefig('fig_volume_vs_rate.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.6 Análisis Temporal

In [ ]:
mql['year_month']   = mql['first_contact_date'].dt.to_period('M')
deals['year_month'] = deals['won_date'].dt.to_period('M')

monthly_mql   = mql.groupby('year_month').size()
monthly_deals = deals.groupby('year_month').size()

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

for ax, series, color, title, ylabel in [
    (axes[0], monthly_mql,   COLORS[0], 'MQLs Generados por Mes',           'Número de MQLs'),
    (axes[1], monthly_deals, COLORS[1], 'Conversiones (Won Deals) por Mes',  'Número de Conversiones'),
]:
    x = series.index.astype(str)
    ax.plot(x, series.values, marker='o', color=color, linewidth=2.5, markersize=7)
    ax.fill_between(x, series.values, alpha=0.15, color=color)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=45)
    for xi, yi in zip(x, series.values):
        ax.annotate(str(yi), (xi, yi), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('fig_monthly_trends.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Pico de generación de MQLs:  {monthly_mql.idxmax()}  ({monthly_mql.max()} MQLs)')
print(f'Pico de conversiones:         {monthly_deals.idxmax()} ({monthly_deals.max()} deals)')

## 2.7 Análisis del Tiempo de Conversión

In [ ]:
merged = mql.merge(deals[['mql_id', 'won_date']], on='mql_id', how='inner')
merged['lead_time_days'] = (merged['won_date'] - merged['first_contact_date']).dt.days

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución completa
axes[0].hist(merged['lead_time_days'], bins=50, color=COLORS[3], edgecolor='white', linewidth=0.5)
axes[0].axvline(x=merged['lead_time_days'].median(), color='red', linestyle='--',
                label=f'Mediana: {merged["lead_time_days"].median():.0f} días')
axes[0].axvline(x=90, color='orange', linestyle=':', linewidth=2, label='Umbral 90 días')
axes[0].set_xlabel('Días primer contacto → conversión')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución del Tiempo de Conversión (completo)')
axes[0].legend()

# Zoom 0-180 días
subset = merged[merged['lead_time_days'].between(0, 180)]
axes[1].hist(subset['lead_time_days'], bins=36, color=COLORS[4], edgecolor='white', linewidth=0.5)
axes[1].axvline(x=subset['lead_time_days'].median(), color='red', linestyle='--',
                label=f'Mediana: {subset["lead_time_days"].median():.0f} días')
axes[1].set_xlabel('Días')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Zoom: 0 a 180 días')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_lead_time.png', dpi=120, bbox_inches='tight')
plt.show()

print('Estadísticas del tiempo de conversión (días):')
print(merged['lead_time_days'].describe().round(1))
print(f'\nLeads con tiempo NEGATIVO: {(merged["lead_time_days"] < 0).sum()} '
      f'({(merged["lead_time_days"] < 0).mean()*100:.1f}%)')
print(f'Leads con journey > 90 días: {(merged["lead_time_days"] > 90).sum()} '
      f'({(merged["lead_time_days"] > 90).mean()*100:.1f}%)')

## 2.8 Análisis de Segmentos en Deals Convertidos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

segments = [
    ('business_segment',      'Segmento de Negocio'),
    ('lead_type',             'Tipo de Lead'),
    ('lead_behaviour_profile','Perfil de Comportamiento'),
]

for ax, (col, title) in zip(axes, segments):
    col_data = deals[col].fillna('(not set)').replace('', '(not set)')
    counts = col_data.value_counts().head(10)
    bars = ax.barh(counts.index, counts.values, color=COLORS[5])
    ax.set_title(title)
    ax.set_xlabel('Número de deals')
    for bar, val in zip(bars, counts.values):
        ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig_segments.png', dpi=120, bbox_inches='tight')
plt.show()

## ✅ Hallazgos Clave — Fase 2

### Volumen y Canales
- **organic_search** domina el volumen (2 296 MQLs, 28.7%), seguido de **paid_search** (19.8%) y **social** (16.9%)
- **paid_search** y **organic_search** generan el mayor número absoluto de conversiones
- **unknown** tiene la mayor tasa de conversión entre canales rastreados (16.3%)
- **email** y **other** tienen las tasas más bajas (3.0 % y 2.7 % respectivamente)

### Calidad de Datos
- 60 registros MQL sin `origin` → etiquetados como `(not set)`
- `has_company`, `has_gtin`, `average_stock` tienen >90 % vacíos → no usables
- 177/842 deals sin `lead_behaviour_profile` → campo descriptivo, no crítico para atribución

### Temporal
- La generación de MQLs crece notablemente a partir de Oct 2017
- Las conversiones siguen con ≈2–3 meses de rezago respecto a la generación de leads

### Tiempo de Conversión
- Mediana: **14 días** | Media: **48.4 días**
- 2 conversiones con tiempo negativo (posible error de captura)
- 150 conversiones (17.8 %) superan los 90 días → se mantienen con flag `long_journey`

### Implicación para el Modelo de Markov
> ⚠️ **Limitación estructural:** el dataset registra **un único touchpoint por lead** (`origin` = canal del primer y único contacto). Esto significa que:
> - Los journeys tendrán la forma `Start → canal → Conversion|Non_Conversion`
> - First-touch, Last-touch y Linear darán el mismo resultado
> - El valor del modelo de Markov estará en el **Removal Effect**: cuánto cae la conversión global al eliminar cada canal

---
# Fase 3: Data Preparation

> **Objetivo:** Construir la estructura de journeys lista para modelar con Cadenas de Markov.

**Pasos:**
1. Unir MQL + Deals por `mql_id`
2. Aplicar reglas de negocio (flags, no filtros duros)
3. Construir journeys en formato `Start > canal > Conversion|Non_Conversion`
4. Construir la matriz de transición
5. Validar coherencia y guardar outputs

## 3.1 Merge de Datasets

In [ ]:
# Left join: conservar todos los MQLs
df = mql.merge(
    deals[['mql_id', 'won_date', 'business_segment', 'lead_type', 'lead_behaviour_profile']],
    on='mql_id',
    how='left'
)

df['converted']      = df['won_date'].notna()
df['lead_time_days'] = (df['won_date'] - df['first_contact_date']).dt.days
df['long_journey']   = df['lead_time_days'].gt(90)

print(f'Shape final: {df.shape}')
print(f'\n  Total MQLs:     {len(df):,}')
print(f'  Convertidos:    {df["converted"].sum():,} ({df["converted"].mean()*100:.2f}%)')
print(f'  No convertidos: {(~df["converted"]).sum():,} ({(~df["converted"]).mean()*100:.2f}%)')
print(f'\nVerificación: conversiones en df == filas en deals: '
      f'{df["converted"].sum() == len(deals)}')
display(df.head())

## 3.2 Limpieza de Origins y Aplicación de Reglas de Negocio

Los origins ya fueron normalizados al cargar los datos (blancos → `(not set)`).  
Aquí registramos los casos edge y los flags aplicados.

In [ ]:
print('Distribución de origins en el dataset unido:')
display(df['origin'].value_counts().to_frame('count'))

neg_time = df[df['converted'] & df['lead_time_days'].lt(0)]
print(f'\nConversiones con tiempo negativo: {len(neg_time)}')
if len(neg_time) > 0:
    display(neg_time[['mql_id', 'origin', 'first_contact_date', 'won_date', 'lead_time_days']])

print(f'\nConversiones con journey > 90 días (long_journey): '
      f'{df.loc[df["converted"], "long_journey"].sum()} '
      f'({df.loc[df["converted"], "long_journey"].mean()*100:.1f}%)')
print('-> Decision: mantener todos los registros. Flag long_journey=True para analisis de sensibilidad en Fase 5.')

## 3.3 Construcción de Journeys

In [ ]:
def make_journey_str(row: pd.Series) -> str:
    endpoint = 'Conversion' if row['converted'] else 'Non_Conversion'
    return f'Start > {row["origin"]} > {endpoint}'

df['journey_str'] = df.apply(make_journey_str, axis=1)

print(f'Journeys únicos: {df["journey_str"].nunique()}')
print('\nFrecuencia de cada journey:')
display(df['journey_str'].value_counts().to_frame('count'))

## 3.4 Construcción de la Matriz de Transición

In [ ]:
# Extraer pares (from_state, to_state) de cada journey
transitions = []
for journey_str in df['journey_str']:
    parts = journey_str.split(' > ')
    for i in range(len(parts) - 1):
        transitions.append({'from_state': parts[i], 'to_state': parts[i + 1]})

trans_df = pd.DataFrame(transitions)

# Conteo de transiciones
trans_counts = (
    trans_df.groupby(['from_state', 'to_state'])
    .size()
    .reset_index(name='count')
)

# Pivot a formato matricial
trans_pivot = (
    trans_counts
    .pivot(index='from_state', columns='to_state', values='count')
    .fillna(0)
    .astype(int)
)

# Normalizar filas → probabilidades de transición
trans_prob = trans_pivot.div(trans_pivot.sum(axis=1), axis=0).round(6)

print('Matriz de transición — CONTEOS:')
display(trans_pivot)
print('\nMatriz de transición — PROBABILIDADES:')
display(trans_prob)

## 3.5 Visualización de la Matriz de Transición

In [ ]:
channels_list = sorted(
    [s for s in trans_prob.index if s not in ('Start', 'Conversion', 'Non_Conversion')]
)
hm_cols = [c for c in ['Conversion', 'Non_Conversion'] if c in trans_prob.columns]
hm_data  = trans_prob.loc[[c for c in channels_list if c in trans_prob.index], hm_cols]

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Heatmap de probabilidades
sns.heatmap(
    hm_data, annot=True, fmt='.3f', cmap='RdYlGn',
    linewidths=0.5, ax=axes[0], vmin=0, vmax=0.30,
    annot_kws={'size': 10}
)
axes[0].set_title('Probabilidades de Transición\n(canal → estado final)')
axes[0].set_xlabel('Estado destino')
axes[0].set_ylabel('Canal de origen')

# Bar chart: P(Conversion) por canal
if 'Conversion' in hm_data.columns:
    conv_probs = hm_data['Conversion'].sort_values(ascending=True)
    bar_colors = [COLORS[0] if v > global_rate / 100 else COLORS[2] for v in conv_probs]
    axes[1].barh(conv_probs.index, conv_probs.values * 100, color=bar_colors)
    axes[1].axvline(x=global_rate, color='red', linestyle='--', alpha=0.8,
                    label=f'Media: {global_rate:.2f}%')
    axes[1].set_xlabel('P(Conversion) %')
    axes[1].set_title('Probabilidad de Conversión por Canal\n(verde = sobre la media)')
    axes[1].legend()
    for i, (ch, val) in enumerate(conv_probs.items()):
        axes[1].text(val * 100 + 0.1, i, f'{val*100:.2f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig_transition_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

## 3.6 Distribución de Tráfico de Entrada (Start → Canal)

In [ ]:
if 'Start' in trans_prob.index:
    start_probs = trans_prob.loc['Start'].dropna().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(start_probs.index, start_probs.values * 100, color=COLORS)
    ax.set_title('Distribución de Tráfico de Entrada por Canal\n(P(Start → Canal))')
    ax.set_xlabel('Canal')
    ax.set_ylabel('Probabilidad (%)')
    for bar, val in zip(bars, start_probs.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9
        )
    plt.tight_layout()
    plt.savefig('fig_start_transitions.png', dpi=120, bbox_inches='tight')
    plt.show()

## 3.7 Validaciones de Calidad

In [ ]:
print('=' * 55)
print('VALIDACIONES DE CALIDAD — FASE 3')
print('=' * 55)

# V1: total conversiones
real_conv   = int(df['converted'].sum())
matrix_conv = int(trans_counts[trans_counts['to_state'] == 'Conversion']['count'].sum())
v1 = real_conv == matrix_conv
print(f'\nV1 — Conversiones reales ({real_conv}) == Conversiones en matriz ({matrix_conv}): '
      f'{"PASS ✓" if v1 else "FAIL ✗"}')

# V2: total transiciones
expected_trans = len(df) * 2  # 2 transiciones por journey (Start>ch, ch>endpoint)
v2 = len(transitions) == expected_trans
print(f'V2 — Total transiciones ({len(transitions)}) == MQLs × 2 ({expected_trans}): '
      f'{"PASS ✓" if v2 else "FAIL ✗"}')

# V3: filas de la matriz de probabilidad suman 1
non_abs = [s for s in trans_prob.index if s not in ('Conversion', 'Non_Conversion')]
row_sums = trans_prob.loc[non_abs].sum(axis=1)
v3 = (row_sums - 1).abs().max() < 1e-6
print(f'V3 — Filas de probabilidad suman 1 (max_dev={((row_sums-1).abs().max()):.2e}): '
      f'{"PASS ✓" if v3 else "FAIL ✗"}')

# V4: sin origins vacíos
v4 = (df['origin'] == '').sum() == 0
print(f'V4 — Sin origins vacíos: {"PASS ✓" if v4 else "FAIL ✗"}')

print(f'\nResumen del dataset listo para modelado:')
print(f'  Total MQLs:         {len(df):,}')
print(f'  Canales únicos:     {df["origin"].nunique()}')
print(f'  Conversiones:       {df["converted"].sum():,} ({df["converted"].mean()*100:.2f}%)')
print(f'  Estados en cadena:  Start + {df["origin"].nunique()} canales + Conversion + Non_Conversion')

## 3.8 Guardar Outputs

In [ ]:
# Dataset de journeys limpio
cols_save = [
    'mql_id', 'first_contact_date', 'landing_page_id', 'origin',
    'won_date', 'converted', 'lead_time_days', 'long_journey', 'journey_str'
]
df[cols_save].to_csv('journeys_dataset.csv', index=False)

# Matrices de transición
trans_counts.to_csv('transition_counts.csv', index=False)
trans_prob.to_csv('transition_matrix_probs.csv')
trans_pivot.to_csv('transition_matrix_counts.csv')

# Resumen por canal
channel_summary = (
    df.groupby('origin')
    .agg(total=('mql_id', 'count'), conversiones=('converted', 'sum'))
    .assign(tasa_conv_pct=lambda x: (x['conversiones'] / x['total'] * 100).round(2))
    .sort_values('tasa_conv_pct', ascending=False)
)
channel_summary.to_csv('channel_summary.csv')

print('Archivos guardados:')
print('  journeys_dataset.csv         — 8 000 MQLs con journey_str y flags')
print('  transition_counts.csv        — conteo de transiciones entre estados')
print('  transition_matrix_probs.csv  — matriz de probabilidades (input para Fase 4)')
print('  transition_matrix_counts.csv — matriz de conteos')
print('  channel_summary.csv          — resumen de conversión por canal')
print('\nListos para Fase 4: Modelado con Cadenas de Markov + Removal Effect')

## ✅ Entregable Fase 3

### Dataset de Journeys (`journeys_dataset.csv`)
- 8 000 filas, una por MQL
- Formato de journey: `Start > [canal] > Conversion|Non_Conversion`
- Columnas clave: `origin`, `converted`, `lead_time_days`, `long_journey`, `journey_str`

### Matriz de Transición (`transition_matrix_probs.csv`)
- **Start → canal:** distribución de entrada de tráfico por fuente
- **Canal → Conversion:** tasa de conversión por canal (input del Removal Effect)
- **Canal → Non_Conversion:** complemento de conversión
- Estados absorbing: `Conversion`, `Non_Conversion`

### Limitación Confirmada
El dataset registra **un único touchpoint por lead**. Los modelos baseline serán equivalentes. El valor diferencial del modelo de Markov estará en el **Removal Effect** y las simulaciones de escenarios de la Fase 5.

---

### Siguiente Paso — Fase 4: Modelado
1. Baseline: First-touch = Last-touch = Linear (equivalentes con 1 touchpoint)
2. Cadena de Markov orden 1 sobre `transition_matrix_probs.csv`
3. Cálculo del Removal Effect por canal
4. Tabla comparativa de atribución de las 842 conversiones por canal